# Fixed vs. Dynamic Few-Shot — Comparison & Analysis

Compares the fixed-example runs (`results_<model>.json`) against the dynamic-selection runs (`results_<model>_dynamic.json`) for all 3 models, using one shared evaluation code path.

**Needs** (on Drive under `NLP Project/`, or run locally from the repo `Notebooks/` folder):
`Results/` with all 6 results files, `Data/sampled.jsonl`, `Data/dynamic_examples.json`.

No model inference here — runs on CPU. A GPU runtime just makes the BERTScore cell much faster (~2 min vs ~30 min); set `USE_BERTSCORE = False` to skip it entirely.

**Contents**: metric tables → paired significance tests (McNemar, Wilcoxon) → answer-type breakdown → retrieval-quality analysis → flip analysis → cost overhead.

In [ ]:
!pip install rouge-score bert-score

In [ ]:
import json
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from scipy import stats
from statsmodels.stats.contingency_tables import mcnemar
from rouge_score import rouge_scorer
from bert_score import score as compute_bert_score

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
print(f"Running in Colab: {IN_COLAB}")

In [ ]:
MODELS = ["llama", "mistral", "gemma"]
PAIRS = [("few_shot", "dynamic_few_shot"), ("few_shot_cot", "dynamic_few_shot_cot")]
USE_BERTSCORE = True
ANSWER_MARKER = "ANSWER:"

def first_existing(candidates):
    for c in candidates:
        if os.path.exists(c):
            return c
    raise FileNotFoundError(f"None of these paths exist: {candidates}")

RESULTS_DIR = first_existing(["/content/drive/MyDrive/NLP Project/Results", "../Results", "Results"])
DATA_DIR = first_existing(["/content/drive/MyDrive/NLP Project/Data", "../Data", "Data"])
print("Results dir:", RESULTS_DIR)
print("Data dir:   ", DATA_DIR)

In [ ]:
frames = []
for m in MODELS:
    for suffix, source in [("", "fixed"), ("_dynamic", "dynamic")]:
        path = f"{RESULTS_DIR}/results_{m}{suffix}.json"
        part = pd.read_json(path)
        part["model"] = m
        part["source"] = source
        frames.append(part)
df = pd.concat(frames, ignore_index=True)

with open(f"{DATA_DIR}/sampled.jsonl", encoding="utf-8") as f:
    dataset = [json.loads(line) for line in f]
ref_70b = {it["sample_id"]: it["responses"][0]["response"]
           for it in dataset if it.get("responses")}

with open(f"{DATA_DIR}/dynamic_examples.json", encoding="utf-8") as f:
    dynamic_examples = {int(k): v for k, v in json.load(f).items()}
avg_sim = {sid: sum(e["similarity"] for e in exs) / len(exs)
           for sid, exs in dynamic_examples.items()}

print(df.groupby(["model", "source", "technique"]).size())

## Evaluation functions (identical to the model notebooks)

In [ ]:
def normalize_latex(text):
    text = re.sub(r'\\frac\{([^}]+)\}\{([^}]+)\}', r'\1/\2', text)
    text = re.sub(r'\\boxed\{([^}]+)\}', r'\1', text)
    text = re.sub(r'\$+', '', text)
    text = re.sub(r'\\(left|right|quad|qquad|text|mathrm|mathbf|displaystyle)\b', '', text)
    text = re.sub(r'\\{2,}', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def extract_answer(response):
    if ANSWER_MARKER in response:
        return response.split(ANSWER_MARKER)[-1].strip()

    boxed = re.findall(r'\\boxed\{([^}]+)\}', response)
    if boxed:
        return boxed[-1]

    patterns = [
        r'(?:the\s+)?(?:final\s+)?answer\s+is[:\s]+(.+?)(?:\.|$)',
        r'therefore[,:\s]+(.+?)(?:\.|$)',
    ]
    for pattern in patterns:
        match = re.search(pattern, response, re.IGNORECASE | re.DOTALL)
        if match:
            return match.group(1).strip()

    sentences = [s.strip() for s in response.rstrip('.').split('.') if s.strip()]
    return sentences[-1] if sentences else response

def normalize(text):
    text = normalize_latex(text)
    return text.lower().strip().rstrip('.')

In [ ]:
rouge_l_scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

def exact_match(prediction, reference):
    return normalize(extract_answer(prediction)) == normalize(reference)

def contains_match(prediction, reference):
    return normalize(reference) in normalize(prediction)

def compute_rouge_l(prediction, reference):
    return rouge_l_scorer.score(reference, prediction)['rougeL'].fmeasure

def compute_f1_token(prediction, reference):
    pred_tokens = normalize(extract_answer(prediction)).split()
    ref_tokens = normalize(reference).split()
    if not pred_tokens or not ref_tokens:
        return 0.0
    pred_counts = Counter(pred_tokens)
    ref_counts = Counter(ref_tokens)
    common = sum((pred_counts & ref_counts).values())
    if not common:
        return 0.0
    precision = common / len(pred_tokens)
    recall = common / len(ref_tokens)
    return 2 * precision * recall / (precision + recall)

In [ ]:
print("Computing metrics... (the ROUGE-L vs 70B pass takes ~5-10 min on CPU)")
has_ref = df[df["answer_type"] != "no_answer"].copy()
has_ref["extracted_answer"] = has_ref["response"].apply(extract_answer)
has_ref["exact_match"] = has_ref.apply(lambda r: exact_match(r["response"], r["reference_answer"]), axis=1)
has_ref["contains_match"] = has_ref.apply(lambda r: contains_match(r["response"], r["reference_answer"]), axis=1)
has_ref["rouge_l"] = has_ref.apply(lambda r: compute_rouge_l(r["response"], r["reference_answer"]), axis=1)
has_ref["f1_token"] = has_ref.apply(lambda r: compute_f1_token(r["response"], r["reference_answer"]), axis=1)

has_ref["ref_70b"] = has_ref["sample_id"].map(ref_70b)
mask_70b = has_ref["ref_70b"].notna()
has_ref.loc[mask_70b, "rouge_l_vs_70b"] = has_ref[mask_70b].apply(
    lambda r: compute_rouge_l(r["response"], r["ref_70b"]), axis=1
)
print(f"{len(has_ref)} rows with reference answers "
      f"({has_ref['sample_id'].nunique()} questions x techniques x models)")

In [ ]:
if USE_BERTSCORE:
    preds = has_ref["response"].tolist()
    refs = has_ref["reference_answer"].tolist()
    print(f"Computing BERTScore for {len(preds)} pairs...")
    P, R, F1 = compute_bert_score(preds, refs, lang="en", verbose=True, batch_size=64)
    has_ref["bertscore_f1"] = F1.numpy()
    print("Done!")
else:
    print("Skipped (USE_BERTSCORE = False)")

## Metric tables

In [ ]:
metric_cols = ["exact_match", "contains_match", "rouge_l", "f1_token", "rouge_l_vs_70b"]
if USE_BERTSCORE:
    metric_cols.append("bertscore_f1")

summary = has_ref.groupby(["model", "technique"])[metric_cols].mean().round(4)
summary

In [ ]:
def paired_frames(model, fixed_t, dyn_t):
    sub = has_ref[has_ref["model"] == model]
    f_ = sub[sub["technique"] == fixed_t].set_index("sample_id").sort_index()
    d_ = sub[sub["technique"] == dyn_t].set_index("sample_id").sort_index()
    common = f_.index.intersection(d_.index)
    return f_.loc[common], d_.loc[common]

rows = []
for m in MODELS:
    for fixed_t, dyn_t in PAIRS:
        f_, d_ = paired_frames(m, fixed_t, dyn_t)
        for metric in metric_cols:
            fv = f_[metric].dropna().astype(float)
            dv = d_[metric].dropna().astype(float)
            idx = fv.index.intersection(dv.index)
            rows.append({
                "model": m,
                "pair": f"{fixed_t} → {dyn_t}",
                "metric": metric,
                "fixed": fv.loc[idx].mean(),
                "dynamic": dv.loc[idx].mean(),
                "delta": (dv.loc[idx] - fv.loc[idx]).mean(),
            })

deltas = pd.DataFrame(rows).round(4)
deltas.pivot_table(index=["model", "pair"], columns="metric", values="delta").round(3)

## Paired significance tests

Same 82 questions (those with reference answers) under both conditions, per model:

- **McNemar (exact)** on `contains_match` — tests whether the number of questions only dynamic gets right differs from the number only fixed gets right.
- **Wilcoxon signed-rank** on per-question ROUGE-L differences.

With n=82, roughly 15+ discordant questions are needed before p-values can get small — treat direction and consistency across models as more informative than any single p-value.

In [ ]:
test_rows = []
for m in MODELS:
    for fixed_t, dyn_t in PAIRS:
        f_, d_ = paired_frames(m, fixed_t, dyn_t)
        fc = f_["contains_match"].astype(bool)
        dc = d_["contains_match"].astype(bool)

        both = int((fc & dc).sum())
        fixed_only = int((fc & ~dc).sum())
        dyn_only = int((~fc & dc).sum())
        neither = int((~fc & ~dc).sum())
        mcn_p = mcnemar([[both, fixed_only], [dyn_only, neither]], exact=True).pvalue

        diff = (d_["rouge_l"] - f_["rouge_l"]).astype(float)
        try:
            w_p = stats.wilcoxon(diff).pvalue
        except ValueError:  # all differences zero
            w_p = float("nan")

        test_rows.append({
            "model": m,
            "pair": f"{fixed_t} → {dyn_t}",
            "both_correct": both,
            "fixed_only": fixed_only,
            "dynamic_only": dyn_only,
            "neither": neither,
            "mcnemar_p": round(float(mcn_p), 4),
            "rouge_delta_mean": round(diff.mean(), 4),
            "wilcoxon_p": round(float(w_p), 4),
        })

pd.DataFrame(test_rows)

## Breakdown by answer type

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, (fixed_t, dyn_t) in zip(axes, PAIRS):
    sub = has_ref[has_ref["technique"].isin([fixed_t, dyn_t])]
    pf = sub[sub["technique"] == fixed_t].pivot_table(
        index="model", columns="answer_type", values="contains_match", aggfunc="mean")
    pdn = sub[sub["technique"] == dyn_t].pivot_table(
        index="model", columns="answer_type", values="contains_match", aggfunc="mean")
    sns.heatmap(pdn - pf, annot=True, fmt=".2f", center=0, cmap="RdBu_r",
                vmin=-0.3, vmax=0.3, ax=ax)
    ax.set_title(f"Δ contains_match: {dyn_t} − {fixed_t}")
plt.tight_layout()
plt.show()

## Retrieval quality vs. correctness

Does dynamic selection only help when a genuinely similar example exists in the pool? If yes, per-question example similarity should correlate with performance in the dynamic runs — and the benefit over fixed examples should concentrate in the high-similarity questions.

In [ ]:
dyn = has_ref[has_ref["technique"].str.startswith("dynamic")].copy()
dyn["avg_sim"] = dyn["sample_id"].map(avg_sim)

for metric in ["contains_match", "rouge_l"]:
    rho, p = stats.spearmanr(dyn["avg_sim"], dyn[metric].astype(float))
    print(f"Spearman(example similarity, {metric}): rho={rho:.3f}, p={p:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.regplot(data=dyn, x="avg_sim", y="rouge_l",
            scatter_kws={"alpha": 0.3}, line_kws={"color": "red"}, ax=axes[0])
axes[0].set_title("Example similarity vs ROUGE-L (dynamic runs, all models)")
axes[0].set_xlabel("Avg similarity of 3 retrieved examples")
sns.boxplot(data=dyn, x="contains_match", y="avg_sim", ax=axes[1])
axes[1].set_title("Example similarity by correctness (contains_match)")
plt.tight_layout()
plt.show()

# Does the *benefit* of dynamic over fixed concentrate in high-similarity questions?
rows = []
for m in MODELS:
    for fixed_t, dyn_t in PAIRS:
        f_, d_ = paired_frames(m, fixed_t, dyn_t)
        delta = d_["rouge_l"].astype(float) - f_["rouge_l"].astype(float)
        sim = pd.Series({sid: avg_sim[sid] for sid in delta.index})
        rho, p = stats.spearmanr(sim, delta)
        rows.append({"model": m, "pair": f"{fixed_t} → {dyn_t}",
                     "spearman_sim_vs_rouge_gain": round(rho, 3), "p": round(p, 4)})
pd.DataFrame(rows)

## Flip analysis — which questions changed?

In [ ]:
def flips(model, fixed_t, dyn_t):
    f_, d_ = paired_frames(model, fixed_t, dyn_t)
    fc = f_["contains_match"].astype(bool)
    dc = d_["contains_match"].astype(bool)
    gained = sorted(fc.index[(~fc.values) & (dc.values)])
    lost = sorted(fc.index[(fc.values) & (~dc.values)])
    return gained, lost

for m in MODELS:
    for fixed_t, dyn_t in PAIRS:
        gained, lost = flips(m, fixed_t, dyn_t)
        print(f"{m:8s} {fixed_t} → {dyn_t}:")
        print(f"    dynamic gained {len(gained)}: {gained}")
        print(f"    dynamic lost   {len(lost)}: {lost}")

In [ ]:
def show_comparison(sample_id, model, pair=1):
    fixed_t, dyn_t = PAIRS[pair]
    sub = has_ref[(has_ref["model"] == model) & (has_ref["sample_id"] == sample_id)]
    row_f = sub[sub["technique"] == fixed_t].iloc[0]
    row_d = sub[sub["technique"] == dyn_t].iloc[0]

    print("QUESTION:")
    print(row_f["question"][:600])
    print()
    print("REFERENCE:", row_f["reference_answer"])
    print()
    print("RETRIEVED EXAMPLES:")
    for ex in dynamic_examples[sample_id]:
        print(f"  [{ex['similarity']:.3f}] {ex['question'][:130]}")
    print()
    print(f"--- {fixed_t} ({model}) — correct: {row_f['contains_match']} ---")
    print(row_f["response"][:1500])
    print()
    print(f"--- {dyn_t} ({model}) — correct: {row_d['contains_match']} ---")
    print(row_d["response"][:1500])

# Pick sample_ids from the flip lists above, e.g.:
# show_comparison(3, "llama", pair=1)

## Cost overhead

In [ ]:
cost_techs = [t for p in PAIRS for t in p]
cost = df[df["technique"].isin(cost_techs)].groupby(["model", "technique"])[
    ["generation_time", "response_length"]].mean().round(2)
print(cost)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sub = df[df["technique"].isin(cost_techs)]
sns.barplot(data=sub, x="model", y="generation_time", hue="technique",
            hue_order=cost_techs, ax=axes[0])
axes[0].set_title("Avg generation time per question (s)")
sns.barplot(data=sub, x="model", y="response_length", hue="technique",
            hue_order=cost_techs, ax=axes[1])
axes[1].set_title("Avg response length (words)")
for ax in axes:
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## Headline comparison

In [ ]:
plot_metrics = ["contains_match", "rouge_l"]
if USE_BERTSCORE:
    plot_metrics.append("bertscore_f1")

fig, axes = plt.subplots(len(PAIRS), len(plot_metrics),
                         figsize=(5.5 * len(plot_metrics), 8))
for i, (fixed_t, dyn_t) in enumerate(PAIRS):
    for j, metric in enumerate(plot_metrics):
        ax = axes[i][j]
        sub = has_ref[has_ref["technique"].isin([fixed_t, dyn_t])]
        agg = sub.groupby(["model", "technique"])[metric].mean().reset_index()
        sns.barplot(data=agg, x="model", y=metric, hue="technique",
                    hue_order=[fixed_t, dyn_t], ax=ax)
        ax.set_title(f"{metric}", fontsize=10)
        ax.legend(fontsize=7)
plt.suptitle("Fixed vs dynamic few-shot (top: plain few-shot, bottom: CoT)", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
print("HEADLINE: mean per-question delta (dynamic − fixed), questions with references\n")
print(deltas[deltas["metric"].isin(["contains_match", "rouge_l"])]
      .pivot_table(index=["model", "pair"], columns="metric", values="delta")
      .round(3))
print()
print("Interpretation checklist:")
print(" - Consistent sign of deltas across all 3 models?")
print(" - McNemar/Wilcoxon p-values (small n=82 — direction > significance)")
print(" - Which answer types benefit (heatmap)?")
print(" - Does benefit correlate with retrieval similarity?")
print(" - Caveat for the report: dynamic examples differ in BOTH selection and style")
print("   (70B-generated vs hand-written) — the random-selection ablation would isolate this.")